In [1]:
import os

import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()
os.environ["base_url"]


model = ChatOpenAI(
    model=os.environ["model_name"],
    base_url=os.environ["base_url"],
    api_key=os.environ["api_key"],
)

# 链的本质：一个 Runnable

**链就是一个实现了 `Runnable[Input, Output]` 接口的对象**

`|` 对应 `Runnable.__or__`（不是 `__add__`）：每次 `|` 都返回一个**新的 Runnable**（具体是 `RunnableSequence`），所以能无限链式拼接。`RunnableSequence`  在类层次上直接继承自 `Runnable`

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import Runnable, RunnableSequence

prompt = PromptTemplate(
    template="讲一个关于{topic}的笑话，以台湾大学教机器学习的李宏毅老师的风格"
)
parser = StrOutputParser()

In [3]:
print(f"{isinstance(prompt,Runnable)=}  {isinstance(parser,Runnable)=}")
print(
    f"{issubclass(StrOutputParser,Runnable)=}  {issubclass(PromptTemplate,Runnable)=}"
)

isinstance(prompt,Runnable)=True  isinstance(parser,Runnable)=True
issubclass(StrOutputParser,Runnable)=True  issubclass(PromptTemplate,Runnable)=True


In [4]:
chain = prompt | model | parser

print(f"{type(chain)=} \n{chain=} \n{issubclass(type(chain),RunnableSequence)=})")

type(chain)=<class 'langchain_core.runnables.base.RunnableSequence'> 
chain=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='讲一个关于{topic}的笑话，以台湾大学教机器学习的李宏毅老师的风格')
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.12', 'langchain': '1.4.6', 'langchain-openai': '1.7.2'}}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001E4186C4830>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001E4186C52B0>, root_client=<openai.OpenAI object at 0x000001E4186C4050>, root_async_client=<openai.AsyncOpenAI object at 0x000001E4186C5010>, model_name='deepseek-ai/DeepSeek-V4.1-Flash', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.siliconflow.cn/v1', stream_chunk_timeout=120.0)
| StrOutputParser() 
issubclass(type(chain),RunnableSequence)=True)


In [5]:
start = {"topic": "公园象棋"}
print(chain.invoke(start))

好，各位同学，我们今天不讲 gradient descent，来讲一个公园象棋的笑话。

假设你要做一个象棋 AI。你要找一个 function，input 是棋盘，output 是下一步。你收集了十万局棋谱，train 一个很大的 model，loss 一直降，accuracy 99%。你觉得：嗯，这个应该可以挑战公园阿伯了。

你到公园摆好棋盘。阿伯第一手炮二平五，你的 model 说：OK，这个看过。  
阿伯第二手，先喝一口茶，再把炮拿起来，又放回去，说：“我想一下。”  
你的 input 里面没有“喝茶”这个 feature，整个 attention 就乱掉。

阿伯第三手，走了一步你从来没看过。你抗议：“阿伯，这不合棋谱！”  
阿伯说：“棋谱是 training data，不是 ground truth。我赢了才是 ground truth。”

你问阿伯：“你的 policy 怎么 train 的？”  
阿伯说：“我每天来公园，赢了 reward 是隔壁阿嬷说‘水啦’，输了 reward 是‘我刚刚让你的’。这叫 reinforcement learning from human feedback，不过 feedback 只有我。”

你问：“那 learning rate 多少？”  
阿伯说：“我没有 learning rate，我有喝茶 rate。”

最后你输了。阿伯拍拍你说：“少年仔，你这个 model 太 overfitting。公园的 domain 有风、有烟、有阿嬷跳土风舞，你都没有 train。回去 data augmentation 一下。”

所以各位同学，machine learning 的尽头不是 AGI，是公园阿伯。  
好，那这个笑话的 loss 我们就不 backpropagate 了，下课。


# RunnableParallel


`RunnableParallel` 使用 `asyncio.gather` 来同时运行可运行对象

`ainvoke` 会沿链调用每个组件的 `ainvoke`。当执行到 `RunnablePassthrough.assign`（内部是 `RunnableParallel`）时，它会对 `translation1`、`translation2` 两个分支调用 `asyncio.gather`，**并发**执行 qwen 和 kimi 两次翻译，而不是串行。

In [6]:
deepseek_model = ChatOpenAI(
    model=os.environ["model_name"],
    base_url=os.environ["base_url"],
    api_key=os.environ["api_key"],
)

qwen_model = ChatOpenAI(
    model=os.environ["secondary_model_name"],
    base_url=os.environ["base_url"],
    api_key=os.environ["api_key"],
)

kimi_model = ChatOpenAI(
    model=os.environ["tertiary_model_name"],
    base_url=os.environ["base_url"],
    api_key=os.environ["api_key"],
)

In [16]:
from langchain_core.runnables import RunnablePassthrough

poem = "浔阳江头夜送客，枫叶荻花秋瑟瑟"
poem_template = PromptTemplate(template="请将下面的诗句翻译成越南语：{poem}")
judge_template = PromptTemplate(
    template="""
    分析下面两种对中文诗句：{poem}的翻译质量，判断哪种翻译更好，并给出理由。
    翻译1: {translation1},
    翻译2: {translation2}
"""
)

kimi_chain_handler = poem_template | kimi_model | parser
qwen_chain_handler = poem_template | qwen_model | parser
deepseek_chain_summary = judge_template | deepseek_model | parser

# RunnablePassthrough.assign 让第一个变量poem传递给
# kimi_chain_handler和qwen_chain_handler，得到两个翻译结果，
# 然后将这两个翻译结果传递给deepseek_chain_summary进行质量判断。

map_chain = RunnablePassthrough.assign(
    translation1=qwen_chain_handler,
    translation2=kimi_chain_handler,
) | RunnablePassthrough.assign(
    judge=deepseek_chain_summary,
)


In [ ]:
result = await map_chain.ainvoke({"poem": poem})

In [ ]:
from IPython.display import Markdown, display

display(
    Markdown(f"""
## 原句
{result["poem"]}

## 翻译1（Qwen）
{result["translation1"]}

## 翻译2（Kimi）
{result["translation2"]}

## 评判（DeepSeek）
{result["judge"]}
""")
)

**最终结果:** 结论：**翻译2整体更好**，但也不是完全无误；它需要把 **“Sầm Dương”改为“Tầm Dương”**。翻译1的问题更多，且有几处属于较明显的汉越音/用词错误。

### 1. 地名“浔阳”
- “浔阳”的汉越音应为 **Tầm Dương**，不是 **Tuấn Dương**（翻译1），也不是 **Sầm Dương**（翻译2）。
- 所以两者在地名上都有错，但翻译2其他部分准确度明显更高。

### 2. 关键词语
| 中文 | 正确/较好译法 | 翻译1 | 翻译2 |
|---|---|---|---|
| 枫叶 | phong diệp / lá phong | **Phỉ diệp**，错误 | Phong diệp，正确 |
| 荻花 | địch hoa / hoa địch | hoa giách，非标准，至少不如 địch 可靠 | hoa địch / địch hoa，正确 |
| 瑟瑟 | sắt sắt（汉越音）/ xào xạc（拟声） | thu rào rạt，“rào rạt”偏风急水响，不够贴切 | sắt sắt / xào xạc，处理准确 |
| 夜送客 | dạ tống khách（严格汉越）/ đêm tiễn khách | đêm tiễn khách，意译可 | dạ tiễn khách，不够纯汉越；但意译 đêm đưa tiễn khách 可 |

### 3. 风格与方法
翻译2同时给出：
- **汉越音翻译**：贴近原文字词、格律；
- **意译**：符合现代越南语表达；
- 还有词语对照，便于理解。

翻译1虽然也分“古诗感”和“通俗自然”两种，但古诗感版出现了 **“Phỉ diệp”“hoa giách”“Tuấn Dương”** 等问题；通俗版仍保留了错误地名和可疑植物名，整体可信度较低。

### 最佳修订版可作：
**汉越音：**
> Tầm Dương giang đầu dạ tống khách,  
> Phong diệp địch hoa thu sắt sắt.

**意译：**
> Đêm đưa tiễn khách bến sông Tầm Dương,  
> Lá phong hoa địch xào xạc mùa thu.

因此，若必须在两者中选，**翻译2更好**；但应把“Sầm Dương”订正为“Tầm Dương”，严格汉越句中的“tiễn khách”也可改为“tống khách”。